<a href="https://colab.research.google.com/github/mafaiziyas/Wearable-AI-Barbell-Activity-Recognition-and-Rep-Counting-Engine/blob/main/scripts/Data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [199]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Data Preperation

###Data profiling + Understanding data

Meta Data: 5 excercies are tracked including bench press, Deadlift, Overheadpress, Barbell row, Squat. A watch with a gyroscope and an accelormeter tracks the moments associated. There are 5 participants, data is divided amongst each participant by exercise (each csv is semi structured)

In [200]:
import pandas as pd
file1= pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-bench-heavy2-rpe8_MetaWear_2019-01-11T16.10.08.270_C42732BE255C_Accelerometer_12.500Hz_1.4.4.csv")
file2= pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-bench-heavy2-rpe8_MetaWear_2019-01-11T16.10.08.270_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv")
filen= pd.read_csv("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-rest-standing_MetaWear_2019-01-18T18.25.39.382_C42732BE255C_Accelerometer_12.500Hz_1.4.41.csv")

In [201]:
filen.head()

,epoch (ms),time (01:00),elapsed (s),x-axis (g),y-axis (g),z-axis (g)
0,1547918739909,2019-01-19 18:25:39.908,0.00,0.951,0.110,0.193
1,1547918739989,2019-01-19 18:25:39.988,0.08,0.971,0.105,0.196
2,1547918740069,2019-01-19 18:25:40.068,0.16,0.954,0.081,0.196
3,1547918740149,2019-01-19 18:25:40.148,0.24,0.988,0.067,0.233
4,1547918740229,2019-01-19 18:25:40.228,0.32,0.971,0.085,0.242


In [202]:
filen.columns

Index(['epoch (ms)', 'time (01:00)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)',
       'z-axis (g)'],
      dtype='object')

All scattered csv files have 6 columns: "epoch (ms)", "time (01:00)", "elapsed (s)", "x-axis (g)", "y-axis (g)", "z-axis (g)"

* **epoch (ms):** Universal Unix timestamp in milliseconds. Will be used as primary key to merge or sync with other sensor data streams.
* **time (01:00):** Human-readable date and time formatted in a local timezone (UTC+1). Redundant for processing and can be dropped.
* **elapsed (s):** Time passed in seconds since the start of the recording session. Useful for calculating sampling rates or elapsed durations.
* **x-axis (g):** Acceleration along the X-axis in gravitational force units (1g≈9.81 m/s 2 ).
* **y-axis (g):** Acceleration along the Y-axis in g.
* **z-axis (g):** Acceleration along the Z-axis in g.

In [203]:
filen.dtypes

,0
epoch (ms),int64
time (01:00),object
elapsed (s),float64
x-axis (g),float64
y-axis (g),float64
z-axis (g),float64


In [204]:
from glob import glob #allows to find all the pathnames matching a specified pattern
files = glob ("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/*.csv")
files #lists the file paths using glob
len(files) #187 csv files found

187

In [205]:
files[0]

'/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-ohp-heavy_MetaWear_2019-01-14T14.55.42.246_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv'

Since csv are differentiated based on excercise type, participant, category whether its a heavy set or not, i need to extract that data from the file names itself first before merging the individual csv

####Extracting data from file paths itself

In [206]:
data_path = "/content/drive/MyDrive/Fitness Tracker/Raw Data Files/"
f = files[0]
f

'/content/drive/MyDrive/Fitness Tracker/Raw Data Files/A-ohp-heavy_MetaWear_2019-01-14T14.55.42.246_C42732BE255C_Gyroscope_25.000Hz_1.4.4.csv'

In [207]:
participant = f.split("-")[0].replace("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/", "")
label = f.split("-")[1]
category = f.split("-")[2].split("_")[0]

In [208]:
df_1 = pd.read_csv(files[0]) #reading file 1 again

#feature eng
df_1["participant"] = participant
df_1["label"] = label
df_1["category"] = category

df_1 #(voila)

,epoch (ms),time (01:00),elapsed (s),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),participant,label,category
0,1547474142440,2019-01-14T14:55:42.440,0.00,3.049,-4.878,0.244,A,ohp,heavy
1,1547474142480,2019-01-14T14:55:42.480,0.04,2.073,-4.329,-0.183,A,ohp,heavy
2,1547474142520,2019-01-14T14:55:42.520,0.08,-2.073,-2.561,-2.622,A,ohp,heavy
3,1547474142560,2019-01-14T14:55:42.560,0.12,-2.805,-4.207,-1.524,A,ohp,heavy
4,1547474142600,2019-01-14T14:55:42.600,0.16,-3.476,-3.537,1.768,A,ohp,heavy
...,...,...,...,...,...,...,...,...,...
375,1547474157440,2019-01-14T14:55:57.440,15.00,-8.476,2.439,0.793,A,ohp,heavy
376,1547474157480,2019-01-14T14:55:57.480,15.04,-3.415,-2.378,1.585,A,ohp,heavy
377,1547474157520,2019-01-14T14:55:57.520,15.08,-0.976,-0.854,1.463,A,ohp,heavy
378,1547474157560,2019-01-14T14:55:57.560,15.12,4.390,-1.768,-0.183,A,ohp,heavy


####Replicating and looping the code to the rest of the files

In [209]:
acceleratometer_df = pd.DataFrame()
gyroscope_df = pd.DataFrame()
acc_df = 1
gyro_df = 1

for f in files:
  #creating individual df from csv
  participant = f.split("-")[0].replace("/content/drive/MyDrive/Fitness Tracker/Raw Data Files/", "")
  label = f.split("-")[1]
  category = f.split("-")[2].split("_")[0]

  df = pd.read_csv(f)
  #feature eng
  df["participant"] = participant
  df["label"] = label
  df["category"] = category

  #differenciating Accelerometer and Gyroscope data
  if "Accelerometer" in f:
    df["set"]= acc_df
    acc_df += 1
    acceleratometer_df = pd.concat([acceleratometer_df, df]) #pd.concat()
  elif "Gyroscope" in f:
    df["set"]= gyro_df
    gyro_df += 1
    gyroscope_df = pd.concat([gyroscope_df, df])

In [210]:
def set_index_to_epoch (df):
  df["epoch (ms)"] = pd.to_datetime(df["epoch (ms)"], unit= "ms") # fixing the data types
  df.dtypes
  df.index = df["epoch (ms)"]
  return df

acceleratometer_df = set_index_to_epoch(acceleratometer_df)
gyroscope_df = set_index_to_epoch(gyroscope_df)
gyroscope_df.head()

,epoch (ms),time (01:00),elapsed (s),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),participant,label,category,set
epoch (ms),,,,,,,,,,
2019-01-14 13:55:42.440,2019-01-14 13:55:42.440,2019-01-14T14:55:42.440,0.00,3.049,-4.878,0.244,A,ohp,heavy,1
2019-01-14 13:55:42.480,2019-01-14 13:55:42.480,2019-01-14T14:55:42.480,0.04,2.073,-4.329,-0.183,A,ohp,heavy,1
2019-01-14 13:55:42.520,2019-01-14 13:55:42.520,2019-01-14T14:55:42.520,0.08,-2.073,-2.561,-2.622,A,ohp,heavy,1
2019-01-14 13:55:42.560,2019-01-14 13:55:42.560,2019-01-14T14:55:42.560,0.12,-2.805,-4.207,-1.524,A,ohp,heavy,1
2019-01-14 13:55:42.600,2019-01-14 13:55:42.600,2019-01-14T14:55:42.600,0.16,-3.476,-3.537,1.768,A,ohp,heavy,1


In [211]:
gyroscope_df.columns

Index(['epoch (ms)', 'time (01:00)', 'elapsed (s)', 'x-axis (deg/s)',
       'y-axis (deg/s)', 'z-axis (deg/s)', 'participant', 'label', 'category',
       'set'],
      dtype='object')

In [212]:
def drop_dt_columns_in_df(df):
  df = df.drop(['epoch (ms)', 'time (01:00)', 'elapsed (s)'], axis=1)
  return df
gyroscope_df = drop_dt_columns_in_df(gyroscope_df)
acceleratometer_df = drop_dt_columns_in_df(acceleratometer_df)

In [213]:
gyroscope_df.columns

Index(['x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)', 'participant',
       'label', 'category', 'set'],
      dtype='object')

In [214]:
print(acceleratometer_df["set"].unique()) #94 sets
print()
print(gyroscope_df["set"].unique()) # 93 sets

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72
 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94]

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72
 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93]


In [215]:
df_merged = pd.concat([acceleratometer_df.loc[:,['x-axis (g)', 'y-axis (g)', 'z-axis (g)']], gyroscope_df], axis=1) #axis 1 is col wise / horizontal
df = df_merged.copy()
df.head()
df.shape

(69677, 10)

In [216]:
df.dropna().shape

(1119, 10)

In [217]:
#significant amout  of data is dropped when .dropna() called

def NA_percent (df):
  return (df.isna().sum()/df.shape[0])*100
NA_percent(df)

,0
x-axis (g),66.161000
y-axis (g),66.161000
z-axis (g),66.161000
x-axis (deg/s),32.233018
y-axis (deg/s),32.233018
z-axis (deg/s),32.233018
participant,32.233018
label,32.233018
category,32.233018
set,32.233018


###Resampling

gyroscope typically samples faster and has a higher maximum data rate than the accelerometer. Hence more data in gyroscope than accelerometer. Because your gyroscope and accelerometer sample data at different speeds, resampling is the perfect way to force them to use the exact same time intervals so they match up perfectly.

In [218]:
#experimanting different resampling parameters

In [219]:
df1 = df.copy()
df1 = df1.iloc[:1000].select_dtypes(include=['number']).resample(rule="S").mean()
df1
#rule="S" resample by each second and squash everything in to row that contains data which has mean data for that second

/tmp/ipykernel_756/492635917.py:2: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df1 = df1.iloc[:1000].select_dtypes(include=['number']).resample(rule="S").mean()


,x-axis (g),y-axis (g),z-axis (g),x-axis (deg/s),y-axis (deg/s),z-axis (deg/s),set
epoch (ms),,,,,,,
2019-01-11 15:08:04,NaN,NaN,NaN,-9.695500,-1.798500,4.573500,50.0
2019-01-11 15:08:05,-0.002222,0.969333,-0.071222,1.007320,-0.731720,-0.399960,50.0
2019-01-11 15:08:06,-0.099231,0.899462,-0.163846,6.978040,2.417120,-11.256200,50.0
2019-01-11 15:08:07,-0.204750,1.055917,-0.156333,1.317040,-2.083000,2.248840,50.0
2019-01-11 15:08:08,-0.079462,0.920462,-0.085692,-9.826840,-4.697560,8.356160,50.0
...,...,...,...,...,...,...,...
2019-01-11 15:10:18,-0.159308,1.022154,-0.169462,-4.687760,-0.229240,-2.634160,12.0
2019-01-11 15:10:19,-0.136167,0.938417,-0.129417,-4.741400,1.731800,14.529320,12.0
2019-01-11 15:10:20,-0.003769,0.938615,-0.106846,6.426880,-5.836560,-4.370720,12.0


In [220]:
df2 = df.copy()
df2 = df2.iloc[:1000].select_dtypes(include=['number']).resample(rule="200ms").mean()
df2.shape

(689, 7)

In [221]:
df.columns

Index(['x-axis (g)', 'y-axis (g)', 'z-axis (g)', 'x-axis (deg/s)',
       'y-axis (deg/s)', 'z-axis (deg/s)', 'participant', 'label', 'category',
       'set'],
      dtype='object')

In [222]:
df.dropna().shape

(1119, 10)

In [223]:
#resampling

sampling = {'x-axis (g)':"mean", 'y-axis (g)':"mean", 'z-axis (g)':"mean", 'x-axis (deg/s)':"mean",
       'y-axis (deg/s)':"mean", 'z-axis (deg/s)':"mean", 'participant':"last", 'label':"last", 'category':"last", "set":"last"}
df = df.resample(rule="100ms").apply(sampling) #100ms is selected as its the optimal window to consider before start losing critical movement details.
df.shape

(7864289, 10)

In [224]:
df_final = df.dropna()
df_final.shape #17912 rows

(17912, 10)

In [225]:
rename_mapper = {'x-axis (g)':"acc_x (g)", 'y-axis (g)':"acc_y (g)", 'z-axis (g)':"acc_z (g)", 'x-axis (deg/s)':"gyr_x (deg/s)",
       'y-axis (deg/s)':"gyr_y (deg/s)", 'z-axis (deg/s)':"gyr_z (deg/s)", 'participant':"participant", 'label':"label", 'category':"category", "set":"set"}
df_final= df_final.rename(columns = rename_mapper)
df_final.head()

,acc_x (g),acc_y (g),acc_z (g),gyr_x (deg/s),gyr_y (deg/s),gyr_z (deg/s),participant,label,category,set
epoch (ms),,,,,,,,,,
2019-01-11 15:08:05.300,0.0135,0.9770,-0.071,-1.524333,3.069333,4.573000,B,bench,heavy1,50.0
2019-01-11 15:08:05.400,-0.0110,0.9700,-0.086,0.732000,-2.104000,-0.518500,B,bench,heavy1,50.0
2019-01-11 15:08:05.500,0.0080,0.9710,-0.073,-3.292333,-0.081333,3.963667,B,bench,heavy1,50.0
2019-01-11 15:08:05.600,0.0120,0.9960,-0.051,-8.597500,-2.774500,-1.006000,B,bench,heavy1,50.0
2019-01-11 15:08:05.700,-0.0040,0.9595,-0.071,9.999667,1.423000,-1.687000,B,bench,heavy1,50.0


#Train-test split

We are going to use 3 particpants A E C for training, D for Validating, B for testing

In [226]:
df = df_final.copy()
df.head()

,acc_x (g),acc_y (g),acc_z (g),gyr_x (deg/s),gyr_y (deg/s),gyr_z (deg/s),participant,label,category,set
epoch (ms),,,,,,,,,,
2019-01-11 15:08:05.300,0.0135,0.9770,-0.071,-1.524333,3.069333,4.573000,B,bench,heavy1,50.0
2019-01-11 15:08:05.400,-0.0110,0.9700,-0.086,0.732000,-2.104000,-0.518500,B,bench,heavy1,50.0
2019-01-11 15:08:05.500,0.0080,0.9710,-0.073,-3.292333,-0.081333,3.963667,B,bench,heavy1,50.0
2019-01-11 15:08:05.600,0.0120,0.9960,-0.051,-8.597500,-2.774500,-1.006000,B,bench,heavy1,50.0
2019-01-11 15:08:05.700,-0.0040,0.9595,-0.071,9.999667,1.423000,-1.687000,B,bench,heavy1,50.0


In [227]:
train = df.loc[df["participant"].isin(["A","E","C"])]
train.shape

(14145, 10)

In [228]:
val = df.loc[df["participant"].isin(["D"])]
val.shape

(2092, 10)

In [229]:
test = df.loc[df["participant"].isin(["B"])]
test.shape

(1675, 10)

Train: ~79% (14,145 rows) — Plenty of data for your model to learn the intricacies of the barbell movements.
Validation (Val): ~12% (2,092 rows)
Test: ~9% (1,675 rows)

In [230]:
#Exporting intermediate files

train.to_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/train.csv")
val.to_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/val.csv")
test.to_csv("/content/drive/MyDrive/Fitness Tracker/Intermediate Data Files (post Data_preprocessing)/test.csv")

In [231]:
train["label"].unique()

array(['bench', 'ohp', 'squat', 'dead', 'row', 'rest'], dtype=object)